In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

builder = SparkSession.builder \
    .appName("PySparkDemo") \
    .master("local[*]") \
    .config("spark.pyspark.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    ) \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("SparkSession created successfully.")

In [ ]:
sales_data = [
    (101, 1001, "Laptop", "Electronics", 2, 50000.0, "2026-01-10", "Mumbai"),
    (102, 1002, "Mobile", "Electronics", 3, 20000.0, "2026-01-12", "Delhi"),
    (103, 1001, "Mouse", "Accessories", 5, 500.0, "2026-01-15", "Mumbai"),
    (104, 1003, "Chair", "Furniture", 2, 4000.0, "2026-01-20", "Pune"),
    (105, 1004, "Table", "Furniture", 1, 8000.0, "2026-02-01", "Ahmedabad"),
    (106, 1002, "Laptop", "Electronics", 1, 50000.0, "2026-02-05", "Delhi"),
    (107, 1005, "Keyboard", "Accessories", 4, 1000.0, "2026-02-10", "Vadodara"),
    (108, 1003, "Mobile", "Electronics", 2, 20000.0, "2026-02-15", "Pune"),
    (109, 1001, "Chair", "Furniture", 3, 4000.0, "2026-03-01", "Mumbai"),
    (110, 1004, "Mouse", "Accessories", 10, 500.0, "2026-03-05", "Ahmedabad"),
    (111, 1005, "Laptop", "Electronics", 2, 50000.0, "2026-03-10", "Vadodara"),
    (112, 1002, "Table", "Furniture", 2, 8000.0, "2026-03-15", "Delhi")
]

sales_schema = StructType([
    StructField("SalesID", IntegerType(), False),
    StructField("CustomerID", IntegerType(), True),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True),
    StructField("SalesDate", StringType(), True),
    StructField("City", StringType(), True)
])

sales_df = spark.createDataFrame(
    sales_data,
    sales_schema
)

sales_df = sales_df.withColumn(
    "SalesDate",
    F.to_date("SalesDate", "yyyy-MM-dd")
).withColumn(
    "TotalAmount",
    F.col("Quantity") * F.col("Price")
)

sales_df.show(truncate=False)
sales_df.printSchema()

In [ ]:
# Z-Ordering

import os
import time

zorder_path = os.path.abspath(
    "output/delta/zorder_sales"
)

large_sales_df = spark.range(0, 1000000) \
    .withColumnRenamed("id", "SalesID") \
    .withColumn(
        "CustomerID",
        F.col("SalesID") % 10000
    ) \
    .withColumn(
        "ProductID",
        F.col("SalesID") % 500
    ) \
    .withColumn(
        "Amount",
        (F.col("SalesID") % 5000) + 100
    ) \
    .withColumn(
        "SalesDate",
        F.expr("""
            date_add(
                DATE '2026-01-01',
                CAST(SalesID % 365 AS INT)
            )
        """)
    )

large_sales_df.show(5)

In [ ]:
large_sales_df.repartition(40) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(zorder_path)

In [ ]:
delta_table = DeltaTable.forPath(
    spark,
    zorder_path
)

delta_table.detail().select(
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

In [ ]:
import builtins
start_time = time.perf_counter()

result = spark.read \
    .format("delta") \
    .load(zorder_path) \
    .filter(F.col("CustomerID") == 1001) \
    .agg(F.sum("Amount")) \
    .collect()

before_time = time.perf_counter() - start_time

print("Result:", result)
print("Time Before:", builtins.round(before_time, 4), "seconds")

In [ ]:
delta_table.optimize() \
    .executeZOrderBy("CustomerID") \
    .show(truncate=False)

In [ ]:
delta_table.detail().select(
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

In [ ]:
start_time = time.perf_counter()

result = spark.read \
    .format("delta") \
    .load(zorder_path) \
    .filter(F.col("CustomerID") == 1001) \
    .agg(F.sum("Amount")) \
    .collect()

after_time = time.perf_counter() - start_time

print("Result:", result)
print("Time After:", builtins.round(after_time, 4), "seconds")

In [ ]:
print("Before:", builtins.round(before_time, 4))
print("After :", builtins.round(after_time, 4))

improvement = (
    (before_time - after_time) / before_time
) * 100

print(
    "Execution Time Reduction:",
    builtins.round(improvement, 2),
    "%"
)

In [ ]:
delta_table.optimize() \
    .executeZOrderBy(
        "CustomerID",
        "SalesDate"
    )

In [ ]:
result = spark.read \
    .format("delta") \
    .load(zorder_path) \
    .filter((F.col("CustomerID") == 1001) & (F.col("SalesDate").between("2026-01-01", "2026-03-31"))) \
    .agg(F.sum("Amount")) \
    #.collect()

In [ ]:
result.show()

In [ ]:
# Liquid clustering
from pathlib import Path
liquid_path = Path(
    "output/delta/liquid_sales"
).resolve().as_uri()

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS RetailLiquidSales
    (
        SalesID BIGINT,
        CustomerID BIGINT,
        ProductID BIGINT,
        Amount BIGINT,
        SalesDate DATE
    )
    USING DELTA
    CLUSTER BY (CustomerID, SalesDate)
    LOCATION '{liquid_path}'
""")

In [ ]:
large_sales_df.createOrReplaceTempView(
    "LargeRetailSales"
)

spark.sql("""
    INSERT OVERWRITE TABLE RetailLiquidSales
    SELECT
        SalesID,
        CustomerID,
        ProductID,
        Amount,
        SalesDate
    FROM LargeRetailSales
""")

In [ ]:
spark.sql("""
    DESCRIBE DETAIL RetailLiquidSales
""").show(truncate=False)

In [ ]:
spark.sql("""
    OPTIMIZE RetailLiquidSales
""").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT
        CustomerID,
        COUNT(*) AS TotalTransactions,
        SUM(Amount) AS TotalSales
    FROM RetailLiquidSales
    WHERE CustomerID = 1001
    GROUP BY CustomerID
""").show()

In [ ]:
spark.sql("""
    ALTER TABLE RetailLiquidSales
    CLUSTER BY (ProductID, SalesDate)
""")

In [ ]:
spark.sql("""
    OPTIMIZE RetailLiquidSales
""").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT
        ProductID,
        SUM(Amount) AS TotalSales
    FROM RetailLiquidSales
    WHERE ProductID = 101
    GROUP BY ProductID
""").show()

In [30]:
spark.sql("""
    ALTER TABLE RetailLiquidSales
    CLUSTER BY NONE
""")

DataFrame[]